In [1]:
import os
import pandas as pd
from tqdm import tqdm
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

PINECONE_API_KEY = "pcsk_5eB5sf_D4PKMqs9oGJVJ7vXqazVSYKiQMqZNFJqvYjFdtrfrBBHaPfrmFMSWnXTzkmFqeU"

INDEX_NAME = "denialcarc"
CSV_PATH = "card_rarc_codes.csv"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_DIM = 384

d:\dice-project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = SentenceTransformer(MODEL_NAME)

d:\dice-project\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP 

In [3]:
pc = Pinecone(api_key=PINECONE_API_KEY)

if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBED_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(INDEX_NAME)

In [6]:
df = pd.read_csv(CSV_PATH)
df.head()

,RARC Codes,Unnamed: 1
0,M1,X-ray not taken within the past 12 months or n...
1,M2,Not paid separately when the patient is an inp...
2,M3,Equipment is the same or similar to equipment ...
3,M4,Alert: This is the last monthly installment pa...
4,M5,Monthly rental payments can continue until the...


In [8]:
def build_text(row):
    return (
        f"Code: {row['code']} | "
        f"Description: {row['description']}"
    )


In [9]:
texts = [build_text(row) for _, row in df.iterrows()]

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

KeyError: 'code'